Para começar:

In [ ]:
%pip install requests pandas numpy

### Automated Patent Acquisition and Dataset Synchronization
This code block handles the direct interface with the European Patent Office (EPO) servers. The pipeline performs massive data extraction based on two critical stability pillars:

1. **Resilience and Authentication Protocol:** The system automatically manages the OAuth2 token lifecycle. It implements Retry and Throttling mechanisms (courtesy delays) to ensure requests comply with OPS 3.2 API traffic limits, preventing interruptions caused by call overflows.

2. **Dynamic Temporal Segmentation (Slicing):** To bypass the native 2000-result limit per query, the algorithm monitors annual data volume. If the volume exceeds this threshold, the tool partitions the search into monthly windows, ensuring full dataset integrity from 2000 to 2026.

> **Note**: Access credentials (MY_KEY and MY_SECRET) are personal and non-transferable. If the rate of 429 (Too Many Requests) errors increases, the extraction engine will automatically adjust the backoff time between blocks.

In [ ]:
# 1. Import the extraction engine from the local module
from epo_api import extract_epo_patents

# 2. Define EPO API Credentials
# Note: Keep these secure. If the tokens are revoked, generate new ones via the EPO developer portal.
MY_KEY = "own_key" 
MY_SECRET = "own_secret"

# 3. Configure Search Parameters and Execute Extraction
print("Initiating patent extraction pipeline...")

df_patentes = extract_epo_patents(
    consumer_key=MY_KEY,
    consumer_secret=MY_SECRET,
    start_year=2000,
    end_year=2026
    # Uncomment the line below to filter by a specific company/institution.
    # If left commented, the script performs a comprehensive 'Global' landscape search.
    # applicant_filter="Alnylam" 
)

# 4. Validate and Display Results
if not df_patentes.empty:
    print(f"\n[SUCCESS] Successfully extracted {len(df_patentes)} unique patent families.")
    display(df_patentes.head())
else:
    print("\n[WARNING] No data was found for this query, or an extraction error occurred.")
    print("Please check your API quota or adjust the search parameters.")

### Patent Analysis and Chart Generation

This code block performs the visual and quantitative analysis of the extracted dataset. Using the custom `epo_analise` module, it executes two main tasks:

1. **Chronological Evolution (Timeline):** Maps the volume of patent families over time. The analysis uses the **Priority Year** to mark the actual date when the experimental data was established. It also allow us to filter out noise prior to the pivotal siRNA discoveries in mammalian cells (2001).
2. **Competitive Landscape (Top Applicants):** Identifies the leading patent holders. The function applies *Entity Resolution* algorithms to clean names, consolidate subsidiaries, and remove individual inventors, ensuring an accurate corporate ranking.

> **Note:** The results (tables and charts) will be automatically exported as high-resolution `.csv` and `.pdf` files in your current working directory.

In [ ]:
# 1. Imports functions generate_timeline and calculate_top_applicants
from epo_analise import generate_timeline, calculate_top_applicants

# 2. Define Input Data and Analytical Parameters
# Ensure this filename matches the most recent dataset generated by the extraction tool
csv_filename = "dataset_siRNA_Alnylam_2026_2026_20260412_144627.csv" 

# Define the temporal boundary for the timeline analysis.
# This filters out early data noise from decades prior to the main siRNA discoveries.
analysis_start_year = 2000
analysis_end_year = 2026

# Generate the chronological timeline (Aggregation by Priority Year)
generate_timeline(
    file_path=csv_filename, 
    start_year=analysis_start_year, 
    end_year=analysis_end_year
)

# Define the cohort size for the competitive landscape chart (e.g., Top 5, Top 10)
top_applicants_count = 5

# Generate the competitive landscape ranking (Entity Resolution and Consolidation)
calculate_top_applicants(
    file_path=csv_filename, 
    top_n=top_applicants_count
)